# Multi-Algorithm Recommendation System Comparison

**Project:** Comparing Multiple Recommendation Algorithms on E-Commerce Data
**Dataset:** `ecommerce_dataset.csv` (synthetic e-commerce interaction data)

## Objective
Implement and compare **five** recommendation algorithms on the same e-commerce dataset,
then evaluate them with common ranking metrics (Precision@K, Recall@K) plus rating-error
metrics (RMSE) where applicable:

1. Popularity-Based (baseline)
2. User-Based Collaborative Filtering
3. Item-Based Collaborative Filtering
4. Matrix Factorization (Truncated SVD)
5. Content-Based Filtering (TF-IDF)

We finish with a side-by-side comparison table and chart to identify the best-performing
algorithm for this dataset.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
np.random.seed(42)


## 2. Load and Briefly Explore the Dataset

In [ ]:
df = pd.read_csv("ecommerce_dataset.csv")
print("Shape:", df.shape)
print("Users:", df.user_id.nunique(), "| Products:", df.product_id.nunique(), "| Categories:", df.category.nunique())
df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="rating", hue="rating", data=df, palette="crest", legend=False, ax=ax)
ax.set_title("Rating Distribution")
plt.show()


## 3. Train/Test Split

We split interactions 80/20. All algorithms are trained only on the training split and
evaluated against the held-out test interactions, so comparisons are fair.


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

all_users = df.user_id.unique()
all_products = df.product_id.unique()

train_matrix = train_df.pivot_table(index="user_id", columns="product_id",
                                     values="rating", aggfunc="mean")
train_matrix = train_matrix.reindex(index=all_users, columns=all_products).fillna(0)

train_user_items = train_df.groupby("user_id")["product_id"].apply(set).to_dict()
test_user_items = test_df.groupby("user_id")["product_id"].apply(set).to_dict()

print("Train matrix shape:", train_matrix.shape)
print("Train interactions:", len(train_df), "| Test interactions:", len(test_df))


## 4. Evaluation Helper

`precision_recall_at_k` samples users from the test set, generates top-K recommendations
from a given algorithm using only training data, and measures overlap with the user's
held-out (test) interactions.


In [ ]:
def precision_recall_at_k(recommend_fn, k=10, n_sample_users=150, seed=42):
    rng = np.random.RandomState(seed)
    candidate_users = list(test_user_items.keys())
    users = rng.choice(candidate_users, size=min(n_sample_users, len(candidate_users)), replace=False)

    precisions, recalls = [], []
    for u in users:
        relevant = test_user_items.get(u, set())
        if not relevant:
            continue
        recs = set(recommend_fn(u, k))
        if not recs:
            continue
        hits = len(recs & relevant)
        precisions.append(hits / len(recs))
        recalls.append(hits / len(relevant))

    return (np.mean(precisions) if precisions else 0.0,
            np.mean(recalls) if recalls else 0.0)


## 5. Algorithm 1 — Popularity-Based

Ranks products by total historical purchases in the training set. Same list for every
user (minus items they've already interacted with). This is our non-personalized
baseline.


In [ ]:
pop_counts = train_df.groupby("product_id")["purchased"].sum().sort_values(ascending=False)

def popularity_recommend(user_id, k=10):
    seen = train_user_items.get(user_id, set())
    return [p for p in pop_counts.index if p not in seen][:k]

p, r = precision_recall_at_k(popularity_recommend)
print(f"Popularity-Based  ->  Precision@10: {p:.3f} | Recall@10: {r:.3f}")


## 6. Algorithm 2 — User-Based Collaborative Filtering

Finds the most similar users (cosine similarity on rating vectors) and recommends
products those similar users rated highly.


In [ ]:
user_sim = cosine_similarity(train_matrix)
user_sim_df = pd.DataFrame(user_sim, index=train_matrix.index, columns=train_matrix.index)

def user_cf_recommend(user_id, k=10, n_neighbors=20):
    if user_id not in user_sim_df.index:
        return popularity_recommend(user_id, k)
    sims = user_sim_df[user_id].drop(index=user_id)
    top_neighbors = sims.sort_values(ascending=False).head(n_neighbors).index
    scores = train_matrix.loc[top_neighbors].mean(axis=0)
    seen = train_user_items.get(user_id, set())
    scores = scores.drop(index=[p for p in seen if p in scores.index], errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

p, r = precision_recall_at_k(user_cf_recommend)
print(f"User-Based CF     ->  Precision@10: {p:.3f} | Recall@10: {r:.3f}")


## 7. Algorithm 3 — Item-Based Collaborative Filtering

Computes item-item similarity and recommends products similar to items the user has
already interacted with.


In [ ]:
item_sim = cosine_similarity(train_matrix.T)
item_sim_df = pd.DataFrame(item_sim, index=train_matrix.columns, columns=train_matrix.columns)

def item_cf_recommend(user_id, k=10):
    seen = train_user_items.get(user_id, set())
    valid_seen = [p for p in seen if p in item_sim_df.columns]
    if not valid_seen:
        return popularity_recommend(user_id, k)
    scores = item_sim_df[valid_seen].mean(axis=1)
    scores = scores.drop(index=[p for p in seen if p in scores.index], errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

p, r = precision_recall_at_k(item_cf_recommend)
print(f"Item-Based CF     ->  Precision@10: {p:.3f} | Recall@10: {r:.3f}")


## 8. Algorithm 4 — Matrix Factorization (Truncated SVD)

Decomposes the sparse user-item matrix into latent factors and reconstructs a dense
predicted-rating matrix, then recommends the highest predicted, unseen products.


In [ ]:
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(train_matrix)
item_factors = svd.components_.T
pred_matrix = pd.DataFrame(user_factors.dot(item_factors.T),
                            index=train_matrix.index, columns=train_matrix.columns)

def svd_recommend(user_id, k=10):
    if user_id not in pred_matrix.index:
        return popularity_recommend(user_id, k)
    scores = pred_matrix.loc[user_id]
    seen = train_user_items.get(user_id, set())
    scores = scores.drop(index=[p for p in seen if p in scores.index], errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

p, r = precision_recall_at_k(svd_recommend)
print(f"SVD (Matrix Fact) ->  Precision@10: {p:.3f} | Recall@10: {r:.3f}")

# Also compute RMSE on known test ratings vs SVD-predicted ratings
merged = test_df.merge(
    pred_matrix.stack().rename("pred_rating").reset_index().rename(
        columns={"level_1": "product_id"}) if False else pred_matrix.reset_index().melt(
        id_vars="user_id", var_name="product_id", value_name="pred_rating"),
    on=["user_id", "product_id"], how="inner"
)
rmse = np.sqrt(mean_squared_error(merged["rating"], merged["pred_rating"]))
print(f"SVD RMSE on test ratings: {rmse:.3f}")


## 9. Algorithm 5 — Content-Based Filtering

Builds a TF-IDF vector from each product's category and name, then recommends items
similar to the ones a user has already interacted with — independent of other users'
behavior, which helps with cold-start items.


In [ ]:
products_df = df[["product_id", "product_name", "category", "price"]].drop_duplicates().reset_index(drop=True)
products_df["content"] = products_df.category + " " + products_df.product_name

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(products_df.content)
content_sim = cosine_similarity(tfidf_matrix)
content_sim_df = pd.DataFrame(content_sim, index=products_df.product_id, columns=products_df.product_id)

def content_recommend(user_id, k=10):
    seen = train_user_items.get(user_id, set())
    valid_seen = [p for p in seen if p in content_sim_df.columns]
    if not valid_seen:
        return popularity_recommend(user_id, k)
    scores = content_sim_df[valid_seen].mean(axis=1)
    scores = scores.drop(index=[p for p in seen if p in scores.index], errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

p, r = precision_recall_at_k(content_recommend)
print(f"Content-Based     ->  Precision@10: {p:.3f} | Recall@10: {r:.3f}")


## 10. Side-by-Side Comparison

We now run all five algorithms at K = 5, 10, and 20 and collect the results into one
comparison table.


In [ ]:
algorithms = {
    "Popularity": popularity_recommend,
    "User-Based CF": user_cf_recommend,
    "Item-Based CF": item_cf_recommend,
    "SVD (Matrix Factorization)": svd_recommend,
    "Content-Based": content_recommend,
}

results = []
for k in [5, 10, 20]:
    for name, fn in algorithms.items():
        p, r = precision_recall_at_k(fn, k=k, n_sample_users=150)
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0
        results.append({"Algorithm": name, "K": k, "Precision@K": round(p, 3),
                         "Recall@K": round(r, 3), "F1@K": round(f1, 3)})

results_df = pd.DataFrame(results)
results_df


In [ ]:
pivot_precision = results_df.pivot(index="Algorithm", columns="K", values="Precision@K")
pivot_precision = pivot_precision.reindex(algorithms.keys())

fig, ax = plt.subplots(figsize=(9, 5))
pivot_precision.plot(kind="bar", ax=ax, colormap="viridis")
ax.set_title("Precision@K by Algorithm")
ax.set_ylabel("Precision")
ax.set_xlabel("Algorithm")
ax.legend(title="K")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
at_10 = results_df[results_df.K == 10].sort_values("F1@K", ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=at_10, x="F1@K", y="Algorithm", hue="Algorithm", palette="rocket", legend=False, ax=ax)
ax.set_title("Algorithm Ranking by F1@10")
plt.tight_layout()
plt.show()

print("Best algorithm at K=10 (by F1):", at_10.iloc[0]["Algorithm"])


## 11. Coverage Comparison

Coverage measures what fraction of the product catalog each algorithm is capable of
recommending across all users — a diversity/discoverability signal that precision and
recall don't capture.


In [ ]:
def catalog_coverage(recommend_fn, k=10, n_sample_users=150, seed=42):
    rng = np.random.RandomState(seed)
    users = rng.choice(list(train_user_items.keys()), size=min(n_sample_users, len(train_user_items)), replace=False)
    recommended = set()
    for u in users:
        recommended.update(recommend_fn(u, k))
    return len(recommended) / len(all_products)

coverage_results = {name: catalog_coverage(fn, k=10) for name, fn in algorithms.items()}
coverage_df = pd.DataFrame(list(coverage_results.items()), columns=["Algorithm", "Coverage@10"]).sort_values("Coverage@10", ascending=False)
coverage_df


## 12. Conclusion

| Aspect | Best Algorithm |
|---|---|
| Personalization accuracy (Precision/Recall) | Typically **Content-Based** or **Item-Based CF** on this dataset |
| Cold-start friendliness | **Popularity-Based** and **Content-Based** |
| Catalog coverage / diversity | Varies — check `coverage_df` above |
| Scalability to large catalogs | **SVD (Matrix Factorization)** compresses well to latent factors |

**Key takeaways:**
- Popularity-based recommendations are the easiest to compute but the least
  personalized — useful mainly as a fallback for new users/items.
- User-based and item-based collaborative filtering both leverage behavioral signals;
  item-based tends to be more stable when the item catalog is smaller than the user base.
- Matrix factorization (SVD) captures latent taste patterns and scales better to large,
  sparse datasets, at the cost of interpretability.
- Content-based filtering is resilient to the cold-start problem for new products but
  can't capture cross-category taste the way collaborative methods can.

**Recommended approach for production:** a **hybrid** combining collaborative filtering
(for personalization) with content-based signals (for cold-start coverage), as explored
in the companion project *"Building a Recommendation System for E-Commerce."*
